# Agent-to-Agent protocol

Read the [guide](README.md). Install the notebooks, multiagent and partner groups for the full tier. All model calls below replay offline; local A2A transports still use real child processes.


In [ ]:
import os
import sys
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

ROOT = next(
    p
    for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "pyproject.toml").exists() and (p / "data").is_dir()
)
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
os.environ["AWS_EC2_METADATA_DISABLED"] = "true"
os.environ["OTEL_SDK_DISABLED"] = "true"
os.environ["LITELLM_LOCAL_MODEL_COST_MAP"] = "True"
os.environ["MEM0_TELEMETRY"] = "false"
from common.network import offline_network


def run_lesson(entry_point, script):
    """Replay the script offline, on a fresh worker event loop when needed."""

    def work():
        previous = sys.argv
        sys.argv = [script, "--mock"]
        try:
            with offline_network():
                entry_point()
        finally:
            sys.argv = previous

    with ThreadPoolExecutor(max_workers=1) as pool:
        pool.submit(work).result(timeout=120)

## 01_partner_server

Predict the result, then run the [script](examples/01_partner_server.py).


In [ ]:
"""Strands A2A server and agent card: Nimbus Tier 4 lesson.

Install groups: multiagent and partner (see README).
Run: uv run --no-sync python modules/18_a2a/examples/01_partner_server.py --mock
Expected: Lesson complete. Mock fixtures cost $0; Ollama has no API fee.
Live inference uses configured rates and bounded calls. AWS resources created: none.
"""

import json

import httpx

from common.lessons import complete, lesson_settings
from common.partner_client import partner_process
from common.pretty import run_cli


def main() -> None:
    _, cfg = lesson_settings()
    # KEY IDEA: a separate process owns its agent factory, history and HTTP server.
    with partner_process(cfg) as (url, token):
        with httpx.Client(trust_env=False, timeout=10) as client:
            card = client.get(
                url + ".well-known/agent-card.json", headers={"Authorization": "Bearer " + token}
            )
            card.raise_for_status()
            print(json.dumps(card.json(), indent=2))
    print("Partner process stopped.")
    complete()


run_lesson(main, "modules/18_a2a/examples/01_partner_server.py")

## 02_remote_agent

Predict the result, then run the [script](examples/02_remote_agent.py).


In [ ]:
"""Remote invocation and streamed artifacts: Nimbus Tier 4 lesson.

Install groups: multiagent and partner (see README).
Run: uv run --no-sync python modules/18_a2a/examples/02_remote_agent.py --mock
Expected: Lesson complete. Mock fixtures cost $0; Ollama has no API fee.
Live inference uses configured rates and bounded calls. AWS resources created: none.
"""

import asyncio
import json

from common.lessons import complete, lesson_settings
from common.partner_client import ask_partner, partner_process
from common.pretty import run_cli


def main() -> None:
    _, cfg = lesson_settings()
    with partner_process(cfg) as (url, token):
        for streaming in (False, True):
            # KEY IDEA: protocol events differ from local model text deltas.
            result = asyncio.run(ask_partner(url, token, streaming=streaming))
            print(json.dumps(result, indent=2))
    complete()


run_lesson(main, "modules/18_a2a/examples/02_remote_agent.py")

## 03_authenticated_client

Predict the result, then run the [script](examples/03_authenticated_client.py).


In [ ]:
"""Authenticated card and invocation: Nimbus Tier 4 lesson.

Install groups: multiagent and partner (see README).
Run: uv run --no-sync python modules/18_a2a/examples/03_authenticated_client.py --mock
Expected: Lesson complete. Mock fixtures cost $0; Ollama has no API fee.
Live inference uses configured rates and bounded calls. AWS resources created: none.
"""

import asyncio

import httpx

from common.lessons import complete, lesson_settings
from common.partner_client import ask_partner, partner_process
from common.pretty import run_cli


def main() -> None:
    _, cfg = lesson_settings()
    with partner_process(cfg) as (url, token):
        with httpx.Client(trust_env=False, timeout=10) as client:
            print("Anonymous card:", client.get(url + ".well-known/agent-card.json").status_code)
            print("Anonymous RPC:", client.post(url, json={}).status_code)
        # KEY IDEA: ClientConfig carries the HTTP credential; the prompt never sees it.
        print(asyncio.run(ask_partner(url, token)))
    complete()


run_lesson(main, "modules/18_a2a/examples/03_authenticated_client.py")

## 04_framework_interop

Predict the result, then run the [script](examples/04_framework_interop.py).


In [ ]:
"""Real LangGraph partner over A2A: Nimbus Tier 4 lesson.

Install groups: multiagent and partner (see README).
Run: uv run --no-sync python modules/18_a2a/examples/04_framework_interop.py --mock
Expected: Lesson complete. Mock fixtures cost $0; Ollama has no API fee.
Live inference uses configured rates and bounded calls. AWS resources created: none.
"""

import asyncio
import json

from common.lessons import complete, lesson_settings
from common.partner_client import ask_partner, partner_process
from common.pretty import run_cli


def main() -> None:
    _, cfg = lesson_settings()
    # KEY IDEA: the same Strands client calls an actual LangGraph StateGraph adapter.
    with partner_process(cfg, backend="langgraph") as (url, token):
        print(json.dumps(asyncio.run(ask_partner(url, token, streaming=True)), indent=2))
    complete()


run_lesson(main, "modules/18_a2a/examples/04_framework_interop.py")

## partner_upgrade

Predict the result, then run the [script](use_case/partner_upgrade.py).


In [ ]:
"""Concierge calls a separate airline partner: Nimbus Tier 4 lesson.

Install groups: multiagent and partner (see README).
Run: uv run --no-sync python modules/18_a2a/use_case/partner_upgrade.py --mock
Expected: Lesson complete. Mock fixtures cost $0; Ollama has no API fee.
Live inference uses configured rates and bounded calls. AWS resources created: none.
"""

import asyncio

from strands import tool

from common.coordination import worker
from common.lessons import complete, lesson_settings
from common.partner_client import ask_partner, partner_process
from common.pretty import run_cli
from common.testing import tool_message


def main() -> None:
    _, cfg = lesson_settings()
    with partner_process(cfg) as (url, token):
        used = []

        @tool
        async def partner_upgrade() -> dict:
            """Quote one reviewed seat upgrade; cannot purchase or change flights."""
            if used:
                raise ValueError("Partner quote already requested")
            used.append(True)
            return (await ask_partner(url, token))["offer"]

        agent = worker(
            cfg,
            "concierge",
            [
                tool_message("partner_upgrade", {}),
                "UP-001: optional seat upgrade $75; nothing purchased.",
            ],
            tools=[partner_upgrade],
            prompt="Call partner_upgrade once; report its quote.",
        )

        async def invoke():
            async with asyncio.timeout(25):
                return await agent.invoke_async("Quote the FLT-0001 upgrade", limits={"turns": 3})

        result = asyncio.run(invoke())
        complete(result, cfg)
        print("Metrics above cover the concierge only. Partner tokens are unavailable over A2A.")
    complete()


run_lesson(main, "modules/18_a2a/use_case/partner_upgrade.py")

## Practice

Complete the [exercises](README.md#exercises) and [quiz](quiz.md). Module 17 defaults to rejection; use the standalone --decision ask command to review and approve a draft.
